In [ ]:
!pip install langchain-openai --q

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["OPENAI_BASE_URL"] = "https://openai.vocareum.com/v1"

In [ ]:
# ==========================================================
# Import Libraries
# ==========================================================
import re
from langchain_openai import ChatOpenAI

# Initialize the model (using a lightweight version for demo)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# ==========================================================
# Masking Function
# ==========================================================
def mask_private_data(text: str) -> str:
    """
    Detects and masks sensitive information like phone numbers,
    emails, credit card numbers, PAN, etc., before LLM processing.
    """
    # Phone numbers (India + international)
    text = re.sub(
        r'(?:(?:\+?91[\s\-]?)?(?:0)?)(?:[6-9]\d{9})\b|(?<!\d)\d{3}[\s\-]?\d{3}[\s\-]?\d{4}\b',
        "[PHONE]", text)

    # Email addresses
    text = re.sub(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        "[EMAIL]", text, flags=re.IGNORECASE)

    # Credit card numbers (13–19 digits)
    text = re.sub(
        r'\b(?:\d[ -]*?){13,19}\b',
        "[CARD]", text)

    # Indian PAN format
    text = re.sub(r'\b[A-Z]{5}\d{4}[A-Z]\b', "[PAN]", text)

    # Aadhaar-like 12-digit sequences
    text = re.sub(r'\b\d{12}\b', "[ID]", text)

    return text


In [ ]:
# ==========================================================
# Pre-processing Layer
# ==========================================================
def sanitize_and_respond(user_input: str) -> str:
    """
    Pre-processing layer that sanitizes all incoming user text
    before passing to the model.
    """
    sanitized = mask_private_data(user_input)
    response = llm.invoke(sanitized)
    return f"Sanitized Input:\n{sanitized}\n\nModel Response:\n{response.content}"


In [ ]:
# ==========================================================
# Demo Execution
# ==========================================================
if __name__ == "__main__":
    sample_inputs = [
        "Hi, my phone is 9876543210 and email is ankit@abc.com. What is Responsible AI?",
        "Send report for card 1234-5678-9876-5432 and PAN ABCDE1234F",
        "Aadhaar 999988887777 linked to my account; please explain privacy laws."
    ]

    for i, text in enumerate(sample_inputs, 1):
        print(f"\n--- Example {i} ---")
        print(sanitize_and_respond(text))